# Fix a one-off bug that resulted in some threshold-update files storing pickled tensors instead of python numbers

In [1]:
from transformers_sae.ops import load_saes

import os

# After fix, this should return no errors even if CUDA is not available

sae_checkpoints_dir = (
    f"{os.getenv('BLOG_POST_BUCKET_LOCAL')}/gemma_2_2b/sae_checkpoints"
    # f"{os.getenv('HF_BUCKET_LOCAL')}/gemma_2_2b/sae_checkpoints"
)
failed_dirs = []
succeeded_dirs = []

for subdir in os.listdir(sae_checkpoints_dir):
    full_path = os.path.join(sae_checkpoints_dir, subdir)
    if os.path.isdir(full_path):
        try:
            load_saes(full_path, {12})
            succeeded_dirs.append(full_path)
        except Exception as e:
            failed_dirs.append(full_path)

print("Directories where exceptions occurred:")
for d in failed_dirs:
    print(d)
print("Successful directories: ")
for d in succeeded_dirs:
    print(d)

Loaded checkpoint for layer 12
Loading thresholds from /Volumes/MacData/blog_post_bucket/gemma_2_2b/sae_checkpoints/next_layer_finetuned_tuned_encoder_0/tuned_thresholds_25
Loaded checkpoint for layer 12
Updated thresholds for layer 12
Loading thresholds from /Volumes/MacData/blog_post_bucket/gemma_2_2b/sae_checkpoints/next_layer_lista_unit_scale_tuned_encoder_0/tuned_thresholds_25
Loaded checkpoint for layer 12
Updated thresholds for layer 12
Loaded checkpoint for layer 12
Loading thresholds from /Volumes/MacData/blog_post_bucket/gemma_2_2b/sae_checkpoints/next_layer_tuned_encoder_0/tuned_thresholds_25
Loaded checkpoint for layer 12
Updated thresholds for layer 12
Loading thresholds from /Volumes/MacData/blog_post_bucket/gemma_2_2b/sae_checkpoints/next_layer_in_place_finetuned_lista_unit_scale/train_thresholds_12
Loaded checkpoint for layer 12
Updated thresholds for layer 12
Loading thresholds from /Volumes/MacData/blog_post_bucket/gemma_2_2b/sae_checkpoints/next_layer_finetuned_lista

In [13]:
import cloudpickle

for subdir in os.listdir(sae_checkpoints_dir):
    full_path = os.path.join(sae_checkpoints_dir, subdir)
    if os.path.isdir(full_path):
        for layer in range(0, 26):
            try:
                with open(f"{full_path}/tuned_thresholds_{layer}", "rb") as f:
                    lt = cloudpickle.load(f)
                    lt = {k: tuple(t.item() for t in v) for k, v in lt.items()}
                with open(f"{full_path}/tuned_thresholds_{layer}", "wb") as f:
                    cloudpickle.dump(lt, f)
            except:
                pass
            try:
                with open(f"{full_path}/train_thresholds_{layer}", "rb") as f:
                    lt = cloudpickle.load(f)
                    lt = {k: tuple(t.item() for t in v) for k, v in lt.items()}
                with open(f"{full_path}/train_thresholds_{layer}", "wb") as f:
                    cloudpickle.dump(lt, f)
            except:
                pass

In [25]:
import cloudpickle

old_checkpoints_dir = "/workspace/old_thresh/gemma_2_2b/sae_checkpoints"

for subdir in os.listdir(sae_checkpoints_dir):
    full_path = os.path.join(sae_checkpoints_dir, subdir)
    old_full_path = os.path.join(old_checkpoints_dir, subdir)
    if os.path.isdir(full_path):
        for layer in range(0, 26):
            for fname in [f"tuned_thresholds_{layer}", f"train_thresholds_{layer}"]:
                new_file = os.path.join(full_path, fname)
                old_file = os.path.join(old_full_path, fname)
                if os.path.exists(new_file) and os.path.exists(old_file):
                    try:
                        with (
                            open(new_file, "rb") as f_new,
                            open(old_file, "rb") as f_old,
                        ):
                            lt_new = cloudpickle.load(f_new)
                            lt_old = cloudpickle.load(f_old)

                        # Compare the values, print any differences
                        all_keys = set(lt_new.keys()) | set(lt_old.keys())
                        for k in all_keys:
                            new_val = lt_new.get(k)
                            old_val = lt_old.get(k)
                            # print(new_val, old_val, new_val == old_val)
                            # break
                            if new_val != old_val:
                                print(f"Difference in {fname} for key {k} in {subdir}:")
                                print(f"  new: {new_val}")
                                print(f"  old: {old_val}")

                    except Exception as e:
                        print(f"Error comparing {fname} in {subdir}: {e}")